# Fig 5G,H / S11C — UKB case examples (data processing)

Fits in-sample SuSiE, SuSiE-RSS and SuSiE-rRSS (`R_finite = B_sketch`, `R_mismatch = "eb_mix"`) on UKB GWAS loci where a single variant's LD mismatch produces multiple wrong CSs under SuSiE-RSS.

**Outputs** (each case = `list(file, fits = list(insample_res, susie_rss_res, susie_eb_res), z, sketch_ld_top, insample_ld_top)`):
- `figure_data/Fig5E.UKB_Case_Example.rds` — main case (`track_fit = TRUE` for the iteration plots)
- `figure_data/FigS11C.UKB_Case_Examples_Supp.rds` — named list of the four supplementary cases

In [1]:
library(tidyverse)
library(MASS)
library(crayon)
library(pecotmr)
library(susieR)
library(vroom)
library(pgenlibr)
library(Rfast)

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)

# susieR GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)
select <- dplyr::select

# Project utilities
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")

# -----------------------------------------------------------------------------
# susieR build in its own environment so it never clobbers the global one.
#   susie_hc : Haochen multi-panel / R_finite build -> vanilla + eb_mix
# -----------------------------------------------------------------------------
source_build <- function(dir) {
  env <- new.env(parent = globalenv())
  for (f in list.files(dir, pattern = "\\.R$", full.names = TRUE)) sys.source(f, envir = env)
  env
}
susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss

## repo helpers last, so nothing from the sourced builds shadows them
source("../../R/utils.R")

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘MASS’


The following object is masked from ‘package:dplyr’:

    select



Attaching package: ‘crayon’


The following object is masked from ‘package:ggplot2’:

    %+%


Warning message:
“package ‘vroom’ was built under R version 4.5.3”

Attaching package: ‘vroom’


The following objects are masked from ‘package:readr’:

    as.col_spec, col_character, col_date, col_datetime, col_double,
    col_factor, col_guess, col_integer, col_logical, col_numbe

In [2]:
# ---- helpers ----
safe_susie <- function(.fn, .label, ...) {
  tryCatch(
    withCallingHandlers(
      .fn(...),
      warning = function(w) invokeRestart("muffleWarning")
    ),
    error = function(e) {
      cat(sprintf("[FAIL] %s: %s\n", .label, conditionMessage(e)))
      NULL
    }
  )
}

## fit the three models on one case file and keep what the figure needs:
## the fits, the sketch z-scores and the lead variant's LD rows (sketch + in-sample)
fit_case <- function(target, track_fit = FALSE) {
  d <- readRDS(target)

  insample_z  <- d[["insample_z"]]
  insample_LD <- d[["insample_LD"]]
  z_sketch    <- d[["z_sketch"]]
  sketch      <- d[["sketch"]]
  sketch_LD   <- d[["sketch_LD"]]
  n           <- d[["n"]]
  B_sketch    <- d[["B_sketch"]]

  insample_res  <- safe_susie(susie_hc, "insample_res",
                              z = insample_z, R = insample_LD, n = n)
  susie_rss_res <- safe_susie(susie_hc, "susie_rss_res",
                              z = z_sketch, X = sketch, n = n, track_fit = track_fit)
  susie_eb_res  <- safe_susie(susie_hc, "susie_eb_res",
                              z = z_sketch, X = sketch, n = n,
                              R_finite = B_sketch, R_mismatch = "eb_mix", track_fit = track_fit)

  top <- which.max(abs(z_sketch))
  list(
    file            = basename(target),
    fits            = list(insample_res = insample_res, susie_rss_res = susie_rss_res, susie_eb_res = susie_eb_res),
    z               = z_sketch,
    sketch_ld_top   = sketch_LD[top, ],
    insample_ld_top = insample_LD[top, ]
  )
}

case_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/cases/eb_wrong/"

## Fig 5E: bouncing effect (chr12, HLR %)

In [3]:
case_main <- fit_case(file.path(case_dir, "chr12:95606906-96464566.HLR_perc.rds"), track_fit = TRUE)   # was file[12]

case_main$fits$insample_res$sets$cs
case_main$fits$susie_rss_res$sets
case_main$fits$susie_eb_res$sets
case_main$fits$susie_eb_res$R_finite_diagnostics$R_reliability_flag

save_data(case_main, "Fig5E.UKB_Case_Example")

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.


HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.


HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.

HINT: Switching to PIP-based convergence: R uncertainty inflation modifies per-variant SER likelihoods, which prevents a consistent model-level ELBO.



$L1
[1] 1074 1155 1180 1268

,min.abs.corr,mean.abs.corr,median.abs.corr
,<dbl>,<dbl>,<dbl>
L1,1.0000000,1.0000000,1.0000000
L2,1.0000000,1.0000000,1.0000000
L6,1.0000000,1.0000000,1.0000000
L10,1.0000000,1.0000000,1.0000000
L5,0.9995700,0.9995700,0.9995700
L8,0.9990274,0.9993469,0.9990308
L3,0.9984744,0.9994560,0.9995020
L7,0.9954699,0.9977079,0.9975395
L9,0.9178829,0.9178829,0.9178829


,min.abs.corr,mean.abs.corr,median.abs.corr
,<dbl>,<dbl>,<dbl>
L1,0.9756269,0.9861522,0.9871695


[1] FALSE

## Fig S10C: supplementary case examples

`supp_1` (chr22 monocyte %) cannot be detected by SLALOM anyway, since SLALOM requires LD² > 0.6 with the lead; this off variant is out of scope.

In [4]:
supp_files <- c(
  supp_1 = "chr22:17336757-18536672.Monocyte_perc.rds",   # was file[22]
  supp_2 = "chr2:176724104-177224104.Haemoglobin.rds",    # was file[20]
  supp_3 = "chr10:95746336-97374665.WBC_count.rds",       # was file[7]
  supp_4 = "chr6:157842638-158886092.PDW.rds"             # was file[28]
)

cases_supp <- map(supp_files, ~ fit_case(file.path(case_dir, .x)))
map(cases_supp, ~ .x$fits$insample_res$sets$cs)

save_data(cases_supp, "FigS11C.UKB_Case_Examples_Supp")

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.


HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.

HINT: Switching to PIP-based convergence: R uncertainty inflation modifies per-variant SER likelihoods, which prevents a consistent model-level ELBO.

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, 

$supp_1
$supp_1$L1
[1] 1051


$supp_2
$supp_2$L1
[1] 882


$supp_3
$supp_3$L3
 [1] 4649 4656 4662 4663 4665 4667 4670 4672 4675 4677 4678 4679 4680 4682 4685
[16] 4686 4688 4690 4692 4695

$supp_3$L1
 [1] 1285 1341 1345 1356 1362 1384 1393 1402 1416 1445 1447 1451 1454 1459 1515
[16] 1534 1556 1563 1565 1578 1581 1588 1634 1657 1678 1692 1701 1704 1730 1741
[31] 1768 1774 1793 1818 1822 1835 1837 1848 1860 1873 1884 1893 1900 1907 1920
[46] 1925 1939


$supp_4
$supp_4$L2
[1] 2555

$supp_4$L1
[1] 1128 1129